In [20]:
import json
import tiktoken

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader,TextLoader,PyPDFDirectoryLoader
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings


In [9]:
load_dotenv(override=True)

True

In [44]:
cle_extraite = os.getenv("CUAD_KEY")

llm = ChatMistralAI(
    model="mistral-small-latest", 
    temperature=0.0,
    api_key=cle_extraite 
)

In [45]:
os.getenv("CUAD_KEY")

'Mmjuf5RR0GpRmRnVuBaZHD7atxClB7rQ'

# 1 - Loading the PDF , chuncking

In [11]:
pdf_path = "DIABATE YOUSSOUF – Data Scientist & AI Engineer.pdf"
pdf_loader = PyPDFLoader(pdf_path)

In [12]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=20,
    encoding_name="gpt2"
)

In [14]:
chunks = pdf_loader.load_and_split(text_splitter)

In [16]:
len(chunks)

6

In [21]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1290.32it/s]


In [24]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="QA_CV",
    persist_directory="./chroma_db"
)

In [25]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [26]:
retrieved_chunks = retriever.invoke("What is the name of the person in the CV ?")

In [27]:
retrieved_chunks

[Document(metadata={'moddate': '2026-06-06T21:08:17+00:00', 'creationdate': '2026-06-06T21:08:17+00:00', 'title': 'DIABATE YOUSSOUF – Data Scientist & AI Engineer', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'total_pages': 3, 'page_label': '2', 'source': 'DIABATE YOUSSOUF – Data Scientist & AI Engineer.pdf', 'producer': 'Skia/PDF m149', 'page': 1}, page_content='2024)\nDEUG Sciences Mathématiques et Informatique (SMI) – FSM (2021 - 2023)\nCertifications\nTensorFlow Developer Certificate – IBM & Kaggle\nIntelligence Artificielle Générative & RAG – IBM\nAgents IA avec LangChain et LlamaIndex – IBM & LinkedIn Learning\nData Science Specialization –\n06/06/2026 22:08 DIABATE YOUSSOUF – Data Scientist & AI Engineer\nﬁle:///home/diabate/Bureau/docPerso/DIABATE YOUSSOUF – Data Scientist & AI Engineer.html 2/3'),
 Document(metadata={'source': 'DIABATE YOUSSOUF – Data Scientist & AI Engineer.pdf', 'page': 1, 'creationdate'

In [28]:
print(retrieved_chunks[0].page_content)

2024)
DEUG Sciences Mathématiques et Informatique (SMI) – FSM (2021 - 2023)
Certifications
TensorFlow Developer Certificate – IBM & Kaggle
Intelligence Artificielle Générative & RAG – IBM
Agents IA avec LangChain et LlamaIndex – IBM & LinkedIn Learning
Data Science Specialization –
06/06/2026 22:08 DIABATE YOUSSOUF – Data Scientist & AI Engineer
ﬁle:///home/diabate/Bureau/docPerso/DIABATE YOUSSOUF – Data Scientist & AI Engineer.html 2/3


In [50]:
prompt = """You are a helpful assistant for answering questions about a CV. 
        Use the following retrieved chunks to answer the question. If you don't know the answer, say you don't know.
        <context>
        {context}
        </context>
        <question>
        {question}
        </question>
        """

## Retriever

In [35]:
user_question = "What is the phone number of the person in the CV ?"
relevant_document_chunks = retriever.invoke(user_question)
context_list = [chunk.page_content for chunk in relevant_document_chunks]
context_for_query = "\n".join(context_list)

In [36]:
print(context_for_query)

2024)
DEUG Sciences Mathématiques et Informatique (SMI) – FSM (2021 - 2023)
Certifications
TensorFlow Developer Certificate – IBM & Kaggle
Intelligence Artificielle Générative & RAG – IBM
Agents IA avec LangChain et LlamaIndex – IBM & LinkedIn Learning
Data Science Specialization –
06/06/2026 22:08 DIABATE YOUSSOUF – Data Scientist & AI Engineer
ﬁle:///home/diabate/Bureau/docPerso/DIABATE YOUSSOUF – Data Scientist & AI Engineer.html 2/3
2024)
DEUG Sciences Mathématiques et Informatique (SMI) – FSM (2021 - 2023)
Certifications
TensorFlow Developer Certificate – IBM & Kaggle
Intelligence Artificielle Générative & RAG – IBM
Agents IA avec LangChain et LlamaIndex – IBM & LinkedIn Learning
Data Science Specialization –
06/06/2026 22:08 DIABATE YOUSSOUF – Data Scientist & AI Engineer
ﬁle:///home/diabate/Bureau/docPerso/DIABATE YOUSSOUF – Data Scientist & AI Engineer.html 2/3
DIABATE YOUSSOUF
Data Scientist & AI Engineer
Téléphone : +212 774507360 | Email : diabateyoussouf390@gmail.com
Locali

In [37]:
prompt = prompt.format(context=context_for_query, question=user_question)

In [38]:
print(prompt)

You are a helpful assistant for answering questions about a CV. 
        Use the following retrieved chunks to answer the question. If you don't know the answer, say you don't know.
        <context>
        2024)
DEUG Sciences Mathématiques et Informatique (SMI) – FSM (2021 - 2023)
Certifications
TensorFlow Developer Certificate – IBM & Kaggle
Intelligence Artificielle Générative & RAG – IBM
Agents IA avec LangChain et LlamaIndex – IBM & LinkedIn Learning
Data Science Specialization –
06/06/2026 22:08 DIABATE YOUSSOUF – Data Scientist & AI Engineer
ﬁle:///home/diabate/Bureau/docPerso/DIABATE YOUSSOUF – Data Scientist & AI Engineer.html 2/3
2024)
DEUG Sciences Mathématiques et Informatique (SMI) – FSM (2021 - 2023)
Certifications
TensorFlow Developer Certificate – IBM & Kaggle
Intelligence Artificielle Générative & RAG – IBM
Agents IA avec LangChain et LlamaIndex – IBM & LinkedIn Learning
Data Science Specialization –
06/06/2026 22:08 DIABATE YOUSSOUF – Data Scientist & AI Engineer
ﬁle

In [46]:
resp = llm.invoke(prompt)

In [47]:
print(resp.content)

The phone number of the person in the CV is **+212 774507360**.


In [51]:
def RAG(query,llm=llm,prompt_template=prompt):
    relevant_document_chunks = retriever.invoke(query)
    context_list = [chunk.page_content for chunk in relevant_document_chunks]
    context_for_query = "\n".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    response = llm.invoke(prompt)
    return response.content
 

In [53]:
print(RAG("What is the competences  of the person in the CV ?"))

Here are the **competences** of the person from the CV:

### **Technical Skills:**
1. **Data Science & AI:**
   - Machine Learning
   - Deep Learning
   - NLP (Natural Language Processing)
   - Computer Vision
   - Generative AI (LLM, RAG)
   - AI Agents

2. **Frameworks & Libraries:**
   - TensorFlow
   - Keras
   - Scikit-Learn
   - Transformers
   - LlamaIndex
   - LangChain

3. **Languages, Databases & Big Data:**
   - Python (implied by frameworks)
   - SQL (implied by databases)
   - (Other languages not explicitly mentioned)

### **Certifications:**
- TensorFlow Developer Certificate (IBM & Kaggle)
- Generative AI & RAG (IBM)
- AI Agents with LangChain & LlamaIndex (IBM & LinkedIn Learning)
- Data Science Specialization (not specified)


In [55]:
print(RAG("I'm angry, i want to eat"))

I'm here to help with questions about the CV you provided. If you have any specific questions about the content, skills, or experiences listed, feel free to ask! Otherwise, I can't assist with your current request.


In [56]:
print(RAG("Who are you ?"))

I am **Youssouf Diabate**, a **Data Scientist & AI Engineer** with expertise in advanced AI systems, including:

- **Retrieval-Augmented Generation (RAG)** for advanced question-answering.
- **Chatbot development** using **Spring Boot** (AI agents & RAG) for intelligent conversational assistants.
- **Question-Answering (QA)** with **Transformers** for extracting information from text.
- **Sentiment Analysis (NLP)** using both **supervised (ML/DL models)** and **unsupervised (K-Means clustering, embeddings)** approaches.
- **Sign Language Recognition (Computer Vision)** for real-time gesture detection and translation.

### **Education**
- **Master 2 in Data Science & AI** (2024–Present) – Faculty of Sciences, Meknès.
- **Licence d’Excellence in Data Science & AI** (2023–2024) – Faculty of Sciences, Meknès.

### **Skills & Certifications**
- **Deep Learning** (CNN, RNN, NLP) – DeepLearning.AI
- **Data Cleaning & Preprocessing** – Cisco & CodeSignal
- **React** – LinkedIn Learning

### **

In [60]:
print(RAG("Who are you ?"))

I am **Youssouf Diabate**, a **Data Scientist & AI Engineer** with expertise in **Natural Language Processing (NLP), Deep Learning, Computer Vision, and AI-driven systems**. Here’s a summary of my background based on the provided CV:

### **Education**
- **Master 2 in Data Science & AI** (2024–Present) – *Faculté des Sciences, Meknès*
- **Licence d’Excellence in Data Science & AI** (2023–2024) – *Faculté des Sciences, Meknès*
- **Certifications**:
  - *Deep Learning* (CNN, RNN, NLP) – *DeepLearning.AI*
  - *Data Cleaning & Preprocessing* – *Cisco & CodeSignal*
  - *React* – *LinkedIn Learning*

### **Technical Skills**
- **NLP & AI Systems**:
  - **RAG (Retrieval-Augmented Generation)** for advanced QA systems.
  - **Chatbot development** (Spring Boot backend for conversational AI).
  - **Question-Answering (QA)** using Transformers for information extraction.
  - **Sentiment Analysis**: Supervised (ML/DL models) and unsupervised (K-Means clustering with embeddings).
- **Computer Visio

In [61]:
print(RAG("What is her formation in Data Science ?"))

Based on the provided context, her formation in Data Science includes:

- **DEUG Sciences Mathématiques et Informatique (SMI)** – Faculty of Sciences and Mathematics (FSM), from **2021 to 2023**.
- **Data Science Specialization** (mentioned but without a specific date or institution in the provided context).

Additionally, she has certifications in:
- **TensorFlow Developer Certificate** (IBM & Kaggle)
- **Intelligence Artificielle Générative & RAG** (IBM)
- **Agents IA avec LangChain et LlamaIndex** (IBM & LinkedIn Learning)
